# NB12 — Final aggregation: every paper table from one place

**Why this notebook exists.** After the 2026-10 fixes, the paper's numbers come from several notebooks
(01, 04, 05, 06, 07, 08, 09, 10, 11). This notebook reads their saved outputs — it trains nothing — and produces:

| Output | Paper use |
|---|---|
| `tables/headline_{dataset}.csv/.tex` | Main results tables (TF-IDF, matched-schedule defaults, COA models, DistilBERT) |
| `tables/wilcoxon.csv/.tex` | Corrected 10-seed Wilcoxon table |
| `tables/cv.csv/.tex` | Corrected 5-fold CV table |
| `tables/ablation_{dataset}.csv` | Ablation (all rows on one schedule) |
| `tables/optimizer_*.csv/.tex` + `optimizer_stats.json` | Multi-seed optimizer comparison with pooled tests |
| `tables/adaboost_fullscale.csv`, `tables/finefood_*.csv` | Auxiliary experiments |
| `numbers_for_paper.json` | Every number quoted in the text, in one file |

**Statistics used for the optimizer comparison.** Each (dataset, seed) pair is one block (3 datasets × 4 seeds = 12).
On complete blocks we run a Friedman test across COA/PSO/RS/TPE, then two-sided Wilcoxon signed-rank tests of COA
against each other optimizer with Holm correction. This is done separately for validation fitness (what the
optimizers optimise) and for test F1 (what we care about). A block is taken from ONE linked notebook that holds all
of it (4 searches + 5 final models); the `08-optcmp-*` notebooks are preferred, so no block is counted twice.

**Inputs (Add Input → Notebook output):** `01-preprocessing`, `04-final-training` (required);
`05a-ablation`, `05b-cross-validation`, `06-comparison-new`, `07-adaboost-baseline`,
`08-optcmp-amazon`, `08-optcmp-yelp`, `08-optcmp-twitter` and, if they hold finished blocks, the former `08a/08b/08c-optcmp-*`,
`09-amazon-fine-food-coa-adaboost`, `10-default-matched-schedule`, `11-distilbert-baseline` (each optional —
a missing input just skips its section and is listed at the end). CPU is enough.

In [ ]:
KAGGLE_USER = "abhaysharma01702"
SLUGS = {   # change a value if you ran a "-part2" copy of a notebook
    "01": "01-preprocessing", "04": "04-final-training", "05a": "05a-ablation", "05b": "05b-cross-validation",
    "06": "06-comparison-new", "07": "07-adaboost-baseline",
    "09": "09-amazon-fine-food-coa-adaboost", "10": "10-default-matched-schedule",
    "11": "11-distilbert-baseline",
}
DATASETS = ["amazon", "yelp", "twitter"]

import os, re, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon

def nb(key):
    return f"/kaggle/input/notebooks/{KAGGLE_USER}/{SLUGS[key]}"

# Optimizer comparison: every linked input holding saved_results/optcmp/ is a source (the notebooks
# 08-optcmp-<dataset>, and the former 08a–f for blocks they finished). The 08-optcmp-* notebooks come first.
def source_name(d):
    return d.replace("\\", "/").split("/input/")[-1].split("/saved_results")[0]
OPTCMP_DIRS = sorted({os.path.dirname(os.path.dirname(f)) for f in
                      glob.glob("/kaggle/input/**/saved_results/optcmp/*/*_s*.json", recursive=True)},
                     key=lambda d: (0 if "08-optcmp-" in source_name(d) else 1, source_name(d)))
print("optimizer-comparison sources linked:", [source_name(d) for d in OPTCMP_DIRS])

for key in ["01", "04"]:   # everything else is optional
    if not os.path.isdir(nb(key)):
        raise FileNotFoundError(f"Required input '{SLUGS[key]}' is not linked (Add Input → Your Work → Notebooks).")

OUT = "/kaggle/working/aggregate"
os.makedirs(f"{OUT}/tables", exist_ok=True)
NUMBERS, MISSING = {}, []

def to_builtin(o):
    """json.dump helper for NumPy scalars/arrays."""
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

def jload(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)

def to_latex(df, path, floatfmt="{:.4f}"):
    """Writes tabular body rows (no preamble) so they can be pasted into the existing table environments."""
    def tex(s):
        return str(s).replace("±", r"$\pm$").replace("%", r"\%").replace("&", r"\&").replace("_", r"\_")
    lines = [" & ".join(tex(c) for c in df.columns) + r" \\ \midrule"]
    for _, row in df.iterrows():
        cells = [floatfmt.format(v) if isinstance(v, (float, np.floating)) else tex(v) for v in row.values]
        lines.append(" & ".join(cells) + r" \\")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")

METRICS = ["F1-score", "Macro-F1", "MCC", "AUC-ROC"]

In [ ]:
# ── 1. Headline tables (defaults from NB10, COA from NB04, DistilBERT from NB11) ──
for ds in DATASETS:
    rows = []
    base = jload(f"{nb('01')}/saved_results/{ds}/baseline_metrics.json")
    for name in ["TF-IDF + SVM", "TF-IDF + LR"]:
        rows.append({"Model": name, **{m: base[name][m] for m in METRICS}})
    for arch, fname in [("ANN", "ann"), ("CNN+BiLSTM", "cnn")]:
        matched = jload(f"{nb('10')}/saved_results/{ds}/default_matched_{fname}_metrics.json")
        if matched:
            rows.append({"Model": f"Default {arch}", **{m: matched[m] for m in METRICS}})
        else:
            MISSING.append(f"NB10 matched default {arch} ({ds}) — falling back to NB01 50/20-epoch default")
            rows.append({"Model": f"Default {arch} (50/20-epoch, superseded)", **{m: base[f"Default {arch}"][m] for m in METRICS}})
        coa = jload(f"{nb('04')}/saved_results/{ds}/{fname}_coa_metrics.json")
        rows.append({"Model": f"COA-{arch}", **{m: coa[m] for m in METRICS}})
    bert = jload(f"{nb('11')}/saved_results/{ds}/distilbert_metrics.json")
    if bert:
        rows.append({"Model": "DistilBERT (untuned)", **{m: bert[m] for m in METRICS}})
    else:
        MISSING.append(f"NB11 DistilBERT ({ds})")
    df = pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
    df.to_csv(f"{OUT}/tables/headline_{ds}.csv", index=False)
    to_latex(df, f"{OUT}/tables/headline_{ds}.tex")
    NUMBERS[f"headline_{ds}"] = df.set_index("Model").to_dict(orient="index")
    print(f"\n{ds.upper()}\n{df.to_string(index=False)}")

In [ ]:
# ── 2. Corrected 10-seed Wilcoxon (NB06) ──────────────────────────────────────
rows = []
for ds in DATASETS:
    for arch in ["ann", "cnn"]:
        w = jload(f"{nb('06')}/saved_results/{ds}/{arch}_wilcoxon.json")
        if not w:
            MISSING.append(f"NB06 Wilcoxon {arch} ({ds})"); continue
        diff = np.array(w["coa_f1"]) - np.array(w["def_f1"])
        rows.append({"Dataset": ds.capitalize(), "Model": arch.upper(), "n seeds": w.get("n_seeds", len(diff)),
                     "COA F1": f"{w['coa_mean']:.4f} ± {w['coa_std']:.4f}",
                     "Default F1": f"{w['def_mean']:.4f} ± {w['def_std']:.4f}",
                     "Mean diff": round(float(diff.mean()), 4), "COA wins": f"{int((diff > 0).sum())}/{len(diff)}",
                     "p (two-sided)": round(float(w["p_value"]), 4)})
if rows:
    wdf = pd.DataFrame(rows); wdf.to_csv(f"{OUT}/tables/wilcoxon.csv", index=False)
    to_latex(wdf, f"{OUT}/tables/wilcoxon.tex"); NUMBERS["wilcoxon"] = rows
    print(wdf.to_string(index=False))

In [ ]:
# ── 3. Corrected CV (NB05) and ablation (NB05) ────────────────────────────────
rows = []
for ds in DATASETS:
    for arch in ["ann", "cnn"]:
        s = jload(f"{nb('05b')}/saved_results/{ds}/{arch}_cv_summary.json")
        if not s:
            MISSING.append(f"NB05b CV {arch} ({ds})"); continue
        sm = s["summary"]
        rows.append({"Dataset": ds.capitalize(), "Model": f"COA-{arch.upper()}",
                     "F1": f"{sm['f1']['mean']:.4f} ± {sm['f1']['std']:.4f}",
                     "MCC": f"{sm['mcc']['mean']:.4f} ± {sm['mcc']['std']:.4f}",
                     "AUC": f"{sm['auc']['mean']:.4f} ± {sm['auc']['std']:.4f}"})
if rows:
    cdf = pd.DataFrame(rows); cdf.to_csv(f"{OUT}/tables/cv.csv", index=False)
    to_latex(cdf, f"{OUT}/tables/cv.tex"); NUMBERS["cv"] = rows
    print(cdf.to_string(index=False))

for ds in DATASETS:
    parts = []
    for arch in ["ann", "cnn"]:
        p = f"{nb('05a')}/saved_results/{ds}/{arch}_ablation.csv"
        if os.path.exists(p):
            parts.append(pd.read_csv(p)[["Model"] + METRICS].assign(Architecture=arch.upper()))
        else:
            MISSING.append(f"NB05a ablation {arch} ({ds})")
    if parts:
        adf = pd.concat(parts); adf.to_csv(f"{OUT}/tables/ablation_{ds}.csv", index=False)
        NUMBERS[f"ablation_{ds}"] = adf.to_dict(orient="records")

In [ ]:
# ── 4. Optimizer comparison (NB08): per-dataset tables + pooled tests ─────────
# A (dataset, seed) block is used only if one source holds all of it: 4 searches with the same budget + 5 final
# models. The first source (08-optcmp-* first) that holds it wins; blocks are never assembled from two sources.
OPTS = ["COA", "PSO", "RS", "TPE"]
blocks, partial = {}, {}
for d in OPTCMP_DIRS:
    for ds in DATASETS:
        seeds = {int(m.group(1)) for f in glob.glob(f"{d}/{ds}/*_s*.json")
                 for m in [re.search(r"_s(\d+)\.json$", f)] if m}
        for s in sorted(seeds):
            if (ds, s) in blocks:
                continue
            searches = {o: jload(f"{d}/{ds}/search_{o}_s{s}.json") for o in OPTS}
            finals = {o: jload(f"{d}/{ds}/final_{o}_s{s}.json") for o in OPTS + ["DEFAULT"]}
            if all(searches.values()) and all(finals.values()) and len({v["budget"] for v in searches.values()}) == 1:
                blocks[(ds, s)] = {"source": source_name(d), "searches": searches, "finals": finals}
                partial.pop((ds, s), None)
            else:
                partial[(ds, s)] = source_name(d)
for (ds, s), src in sorted(partial.items()):
    MISSING.append(f"NB08 block {ds} / seed {s} is incomplete in {src} and was not used (run 08-optcmp-{ds})")
runs = [f for b in blocks.values() for f in b["finals"].values()]
if not runs:
    MISSING.append("NB08 optimizer comparison (no complete block linked)")
else:
    bdf = pd.DataFrame([{"dataset": ds, "seed": s, "source": b["source"], "budget": b["searches"]["COA"]["budget"]}
                        for (ds, s), b in sorted(blocks.items())])
    bdf.to_csv(f"{OUT}/tables/optimizer_blocks.csv", index=False)
    print(f"{len(blocks)} complete block(s):\n{bdf.to_string(index=False)}\n")
    rdf = pd.DataFrame(runs)
    rdf.to_csv(f"{OUT}/tables/optimizer_all_runs.csv", index=False)
    per_ds = (rdf.groupby(["dataset", "optimizer"])
                 .agg(n=("seed", "count"), test_f1_mean=("F1-score", "mean"), test_f1_std=("F1-score", "std"),
                      val_f1_mean=("best_val_f1", "mean"), val_f1_std=("best_val_f1", "std"))
                 .round(4).reset_index())
    per_ds.to_csv(f"{OUT}/tables/optimizer_per_dataset.csv", index=False)
    print(per_ds.to_string(index=False))

    def holm(pvals):
        order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); running = 0.0
        for rank, i in enumerate(order):
            running = max(running, min(1.0, (m - rank) * pvals[i])); adj[i] = running
        return adj

    stats = {}
    for metric, col in [("validation_fitness", "best_val_f1"), ("test_f1", "F1-score")]:
        piv = rdf[rdf["optimizer"].isin(OPTS)].pivot_table(index=["dataset", "seed"], columns="optimizer",
                                                          values=col)
        absent = [o for o in OPTS if o not in piv.columns]
        if absent:
            stats[metric] = {"note": f"no finished runs for {absent}"}; continue
        piv = piv[OPTS].dropna()
        if len(piv) < 3:
            stats[metric] = {"n_blocks": int(len(piv)), "note": "too few complete blocks"}; continue
        fr = friedmanchisquare(*[piv[o].values for o in OPTS])
        ranks = piv[OPTS].rank(axis=1, ascending=False).mean().round(3).to_dict()
        others = [o for o in OPTS if o != "COA"]
        raw = []
        for o in others:
            d = piv["COA"].values - piv[o].values
            raw.append(float(wilcoxon(piv["COA"].values, piv[o].values).pvalue) if np.any(d != 0) else 1.0)
        adj = holm(np.array(raw))
        stats[metric] = {
            "n_blocks": int(len(piv)), "friedman_stat": round(float(fr.statistic), 4),
            "friedman_p": round(float(fr.pvalue), 6), "mean_rank (1=best)": ranks,
            "coa_vs": {o: {"mean_diff": round(float((piv["COA"] - piv[o]).mean()), 4),
                           "coa_wins": int((piv["COA"] > piv[o]).sum()),
                           "p_raw": round(raw[i], 6), "p_holm": round(float(adj[i]), 6)}
                       for i, o in enumerate(others)},
        }
    with open(f"{OUT}/tables/optimizer_stats.json", "w", encoding="utf-8") as fh:
        json.dump(stats, fh, indent=2, default=to_builtin)
    NUMBERS["optimizer_stats"] = stats
    print(json.dumps(stats, indent=2, default=to_builtin))

    # Pooled anytime curves (validation F1 vs evaluations)
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(16, 4), sharey=False)
    for ax, ds in zip(np.atleast_1d(axes), DATASETS):
        for o in OPTS:
            curves = [b["searches"][o]["best_so_far"] for (bds, _), b in sorted(blocks.items()) if bds == ds]
            if curves:
                L = min(len(cv) for cv in curves)          # equal budgets give equal lengths; guard against mixed budgets
                if any(len(cv) != L for cv in curves):
                    print(f"  note: {ds} {o}: curves of different lengths, truncated to {L} evaluations")
                c = np.array([cv[:L] for cv in curves]); ax.plot(np.arange(1, L + 1), c.mean(0), label=f"{o} (n={len(c)})")
        ax.set_title(ds.capitalize()); ax.set_xlabel("Fitness evaluations"); ax.grid(alpha=0.3)
    np.atleast_1d(axes)[0].set_ylabel("Best validation F1 so far (mean over seeds)")
    np.atleast_1d(axes)[0].legend()
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_optimizer_anytime.pdf", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# ── 5. Auxiliary experiments: AdaBoost full scale (NB07) and Fine Food (NB09) ──
p = f"{nb('07')}/saved_results/amazon/adaboost_full_comparison.csv"
if os.path.exists(p):
    ada = pd.read_csv(p, index_col=0)
    ada.to_csv(f"{OUT}/tables/adaboost_fullscale.csv", index=False)
    NUMBERS["adaboost_fullscale"] = ada[["Model"] + METRICS].to_dict(orient="records")
    print("\nNOTE: NB07 compares against NB01's 50/20-epoch defaults; quote the NB10 defaults in the paper.")
else:
    MISSING.append("NB07 AdaBoost full-scale")

ff = f"{nb('09')}/saved_results/amazon_fine_food"
fm = jload(f"{ff}/final_metrics.json")
fw_check = jload(f"{ff}/wilcoxon_result.json")
if fm and (not fw_check or "n_seeds" not in fw_check.get("ann", {})):
    # Only the fixed NB09 (2026-10-12) writes "n_seeds"; the pre-fix version (5 seeds, early-stopping bug) does not.
    MISSING.append("NB09 output comes from the OLD, pre-fix notebook version (no 'n_seeds' in wilcoxon_result.json) — "
                   "Fine Food section skipped; rerun NB09 with the current code")
    fm = None
if fm:
    fdf = pd.DataFrame([{"Model": k, **{m: v[m] for m in METRICS}} for k, v in fm.items()]) \
            .sort_values("F1-score", ascending=False)
    fdf.to_csv(f"{OUT}/tables/finefood_results.csv", index=False); to_latex(fdf, f"{OUT}/tables/finefood_results.tex")
    NUMBERS["finefood_results"] = fdf.to_dict(orient="records")
    fw = jload(f"{ff}/wilcoxon_result.json")
    if fw: NUMBERS["finefood_wilcoxon"] = {k: {kk: vv for kk, vv in v.items() if kk not in ("coa_f1", "def_f1")}
                                           for k, v in fw.items()}
    for fname in ["ablation_results.csv"]:
        if os.path.exists(f"{ff}/{fname}"):
            pd.read_csv(f"{ff}/{fname}").to_csv(f"{OUT}/tables/finefood_ablation.csv", index=False)
    for fname in ["coa_ann_result.json", "coa_cnn_result.json"]:
        r = jload(f"{ff}/{fname}")
        if r: NUMBERS[f"finefood_{fname[:-5]}"] = r
else:
    MISSING.append("NB09 Fine Food")

In [ ]:
with open(f"{OUT}/numbers_for_paper.json", "w", encoding="utf-8") as fh:
    json.dump(NUMBERS, fh, indent=2, default=to_builtin)
print(f"\nWrote {OUT}/numbers_for_paper.json and tables/:")
for f in sorted(os.listdir(f"{OUT}/tables")): print("  ", f)
print("\nMissing inputs (sections skipped or using fallbacks):" if MISSING else "\nAll inputs present.")
for m in MISSING: print("  -", m)